# 第28章 监督学习与正则化——从逻辑回归到高维因子选择

> **动机先行**: 第27章我们手工筛选了一个反转因子。但真实买方研究面对的是数百个候选因子、几十年的数据——手工流水线迟早撑不住。监督学习把"从特征预测收益"变成一个统一的优化问题; 而金融数据"特征多、历史短"的天生结构, 让**过拟合**成为悬在每个量化研究员头顶的剑。本章的主角**正则化**, 就是拴住这把剑的缰绳。
>
> **量化实战定位**: Lasso 因子筛选是买方研究的标配工具; Ridge 收缩与第14章的 Ledoit-Wolf 协方差收缩一脉相承; 而交叉验证——尤其是金融数据特有的时间泄漏问题——是区分专业回测与自欺欺人的分水岭。

---

## 28.1 动机: 当特征比样本还贵

监督学习的任务形式很简单: 给定特征矩阵 $X$ (每个样本一行) 和标签 $y$, 找到一个函数 $f$ 使得 $f(x_i) \approx y_i$, 并且在**没见过的新数据上依然成立**。后半句才是难点。

量化场景天然恶劣: 预测对象是下个月的收益率, 信噪比极低; 可用的历史只有几十年, 按月切样本不过几百期; 而想试的因子动辄上百个。用第11章的语言说, 这是一个**多重检验**随时发生的环境——只要特征够多, 总能在训练数据里"找到"虚假规律。

本章沿一条主线推进: 先建立最简单的分类器 (逻辑回归), 亲眼目睹过拟合如何发生 (往模型里塞噪声), 再引出解药 (正则化), 最后补上评估方法的纪律 (交叉验证)。所有实验都在真实的 A 股因子面板上进行。

## 28.2 从线性分类到逻辑回归

### 28.2.1 Sigmoid 与交叉熵

预测"下月涨还是跌"是二分类问题。线性回归输出的是连续值, 但分类需要的是一个 $\{0, 1\}$ 的标签或一个概率。**逻辑回归**的做法分两步: 先用线性函数打分 $z = w^\top x + b$, 再通过 **sigmoid 函数**把分数压进 $(0,1)$ 当作概率:

$$
p = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

**为什么用这个函数**: sigmoid 有三个好性质——(1) 输出永远在 $(0,1)$ 之间, 可以直接当概率解读; (2) 单调递增, 分数越高越可能涨的排序不变; (3) 处处可导且导数形式极简 ($\sigma' = \sigma(1-\sigma)$), 使得梯度下降高效。

训练目标是让"模型认为会涨的概率"贴近真实标签。衡量贴近程度的标准选择是**交叉熵损失** (也叫对数损失):

$$
L = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i\ln p_i + (1-y_i)\ln(1-p_i)\Big]
$$

它恰好是伯努利模型的负对数似然——最小化交叉熵就是最大似然。两个值得记住的性质: (1) 它关于 $w$ 是凸的, 优化没有局部极值的坑 (对比: 若直接用 MSE 配 sigmoid, 损失面非凸且饱和区梯度消失); (2) 梯度形式极其优雅, $\partial L_i/\partial z_i = p_i - y_i$: 预测错得越离谱, 推得越用力。

### 28.2.2 用九个因子预测下月涨跌

把第27章的因子动物园扩到 9 个 (新增振幅、流动性、距52周高点、价格位置), 构建月度截面面板: 每行是一只股票在某月末的 9 个标准化因子值, 标签是该股下月是否上涨。

**代码导读**: (1) `build_panel()` 从日线数据构建月度因子面板——每行是一只股票在某月末的9个标准化因子值; (2) 按2025年1月切分为训练集(前)和测试集(后); (3) 用无正则的逻辑回归(C极大≈MLE)拟合并报告准确率与系数。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def build_panel():
    df = pd.read_csv('data/stock_data_50_20210601_20260531.csv')
    df['time'] = pd.to_datetime(df['time'])
    px = df.pivot(index='time', columns='thscode', values='close').sort_index().ffill()
    high = df.pivot(index='time', columns='thscode', values='high').sort_index().ffill()
    low = df.pivot(index='time', columns='thscode', values='low').sort_index().ffill()
    volu = df.pivot(index='time', columns='thscode', values='volume').sort_index().ffill()
    mcap = df.pivot(index='time', columns='thscode', values='market_cap').sort_index().ffill()

    s_idx = pd.Series(px.index)
    mon_end = s_idx.groupby(pd.to_datetime(s_idx).dt.to_period('M')).max().values
    pm = px.loc[mon_end]

    feats = {
        '动量12-1':  px.shift(21)/px.shift(252) - 1,
        '反转3月':   -(px/px.shift(63) - 1),
        '反转1月':   -(px/px.shift(21) - 1),
        '低波动':    -(px.pct_change().rolling(60).std()*np.sqrt(252)),
        '振幅':      (high/low - 1).rolling(20).mean(),
        '流动性':    np.log(volu.rolling(20).mean()),
        '市值':      np.log(mcap.loc[mon_end]).reindex(pm.index),
        '距52周高点': px/px.rolling(252).max() - 1,
        '价格位置':   ((px - px.rolling(252).min()) /
                    (px.rolling(252).max() - px.rolling(252).min())),
    }
    frames = []
    for name, daily in feats.items():
        z = daily.loc[mon_end].apply(lambda s: (s-s.mean())/s.std(), axis=1)
        frames.append(z.stack().rename(name))
    X = pd.concat(frames, axis=1)
    y_ret = pm.pct_change().shift(-1).stack().rename('ret')
    data = X.join(y_ret).dropna()
    data['up'] = (data['ret'] > 0).astype(int)
    return data.reset_index().rename(columns={'time': 'date', 'thscode': 'code'})

FEATS = ['动量12-1','反转3月','反转1月','低波动','振幅','流动性','市值','距52周高点','价格位置']

panel = build_panel()
split_date = pd.Timestamp('2025-01-01')

tr = panel[pd.to_datetime(panel['date']) < split_date]
te = panel[pd.to_datetime(panel['date']) >= split_date]

scaler = StandardScaler().fit(tr[FEATS].values)
Xtr, ytr = scaler.transform(tr[FEATS].values), tr['up'].values
Xte, yte = scaler.transform(te[FEATS].values), te['up'].values

clf = LogisticRegression(C=1e6, max_iter=5000)     # C 极大 ≈ 无正则
clf.fit(Xtr, ytr)

acc_tr = clf.score(Xtr, ytr)
acc_te = clf.score(Xte, yte)
base_tr = max(ytr.mean(), 1-ytr.mean())
base_te = max(yte.mean(), 1-yte.mean())

print("=== 逻辑回归: 用9个因子预测下月涨跌 ===")
print(f"训练集: {len(ytr)} 样本 ({tr['date'].min()} ~ {tr['date'].max()}), "
      f"上涨占比 {ytr.mean()*100:.1f}%")
print(f"测试集: {len(yte)} 样本 ({te['date'].min()} ~ {te['date'].max()}), "
      f"上涨占比 {yte.mean()*100:.1f}%")
print()
print(f"训练准确率 = {acc_tr*100:.2f}%   (基准 {base_tr*100:.2f}%)")
print(f"测试准确率 = {acc_te*100:.2f}%   (基准 {base_te*100:.2f}%)")
print()
coef_df = pd.Series(clf.coef_[0], index=FEATS).sort_values(key=np.abs, ascending=False)
print("标准化系数 (按绝对值排序):")
for name, c in coef_df.items():
    print(f"  {name:<8} {c:+.3f}")

**运行结果**:

```
=== 逻辑回归: 用9个因子预测下月涨跌 ===
训练集: 1550 样本 (2022-06-30 00:00:00 ~ 2024-12-31 00:00:00), 上涨占比 41.7%
测试集: 800 样本 (2025-01-27 00:00:00 ~ 2026-04-30 00:00:00), 上涨占比 49.0%

训练准确率 = 58.52%   (基准 58.32%)
测试准确率 = 51.12%   (基准 51.00%)

标准化系数 (按绝对值排序):
  低波动      +0.232
  振幅       +0.135
  反转3月     +0.108
  动量12-1   -0.079
  距52周高点   +0.072
  流动性      +0.049
  价格位置     +0.039
  市值       +0.022
  反转1月     -0.012
```

**观察**:

1. **信号弱是常态**: 测试准确率只比"无脑猜多数类"高 0.12 个百分点。这不是模型不行, 而是月度收益本来就接近抛硬币 (呼应第27章: IC=0.06 已经算是好因子)。**任何声称月度择股准确率超过 60% 的模型, 第一反应应该是查泄漏**。
2. **系数方向与第27章互相印证**: 低波动的系数最大 (+0.23), 反转3月为正 (+0.11), 动量为负 ($-0.08$)——单因子 IC 分析与多因子联合模型的结论一致。
3. **注意训练/测试的切分方式**: 按**时间**切而非随机切——2025 年之前训练、之后测试。这个细节在 28.5 节会变成生死攸关的原则。

## 28.3 过拟合现场实验

过拟合最可怕的地方在于它**看起来像是在进步**: 训练误差持续下降。做一个残酷的实验——往面板里掺入与市场毫无关系的纯高斯噪声特征, 看模型表现如何变化:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def build_panel():
    df = pd.read_csv('data/stock_data_50_20210601_20260531.csv')
    df['time'] = pd.to_datetime(df['time'])
    px = df.pivot(index='time', columns='thscode', values='close').sort_index().ffill()
    high = df.pivot(index='time', columns='thscode', values='high').sort_index().ffill()
    low = df.pivot(index='time', columns='thscode', values='low').sort_index().ffill()
    volu = df.pivot(index='time', columns='thscode', values='volume').sort_index().ffill()
    mcap = df.pivot(index='time', columns='thscode', values='market_cap').sort_index().ffill()

    s_idx = pd.Series(px.index)
    mon_end = s_idx.groupby(pd.to_datetime(s_idx).dt.to_period('M')).max().values
    pm = px.loc[mon_end]

    feats = {
        '动量12-1':  px.shift(21)/px.shift(252) - 1,
        '反转3月':   -(px/px.shift(63) - 1),
        '反转1月':   -(px/px.shift(21) - 1),
        '低波动':    -(px.pct_change().rolling(60).std()*np.sqrt(252)),
        '振幅':      (high/low - 1).rolling(20).mean(),
        '流动性':    np.log(volu.rolling(20).mean()),
        '市值':      np.log(mcap.loc[mon_end]).reindex(pm.index),
        '距52周高点': px/px.rolling(252).max() - 1,
        '价格位置':   ((px - px.rolling(252).min()) /
                    (px.rolling(252).max() - px.rolling(252).min())),
    }
    frames = []
    for name, daily in feats.items():
        z = daily.loc[mon_end].apply(lambda s: (s-s.mean())/s.std(), axis=1)
        frames.append(z.stack().rename(name))
    X = pd.concat(frames, axis=1)
    y_ret = pm.pct_change().shift(-1).stack().rename('ret')
    data = X.join(y_ret).dropna()
    data['up'] = (data['ret'] > 0).astype(int)
    return data.reset_index().rename(columns={'time': 'date', 'thscode': 'code'})

FEATS = ['动量12-1','反转3月','反转1月','低波动','振幅','流动性','市值','距52周高点','价格位置']

panel = build_panel()
split_date = pd.Timestamp('2025-01-01')
tr = panel[pd.to_datetime(panel['date']) < split_date]
te = panel[pd.to_datetime(panel['date']) >= split_date]
scaler = StandardScaler().fit(tr[FEATS].values)
Xtr, ytr = scaler.transform(tr[FEATS].values), tr['up'].values
Xte, yte = scaler.transform(te[FEATS].values), te['up'].values

import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(28)

print("=== 往面板里掺入纯噪声特征: 训练/测试准确率的分道扬镳 ===")
print(f"{'噪声特征数':>8} | {'总特征数':>7} | {'训练准确率':>9} | {'测试准确率':>9}")
for k_noise in [0, 10, 50, 200, 500]:
    Xn_tr = np.hstack([Xtr, rng.standard_normal((len(Xtr), k_noise))])
    Xn_te = np.hstack([Xte, rng.standard_normal((len(Xte), k_noise))])
    m = LogisticRegression(C=1e6, max_iter=3000)
    m.fit(Xn_tr, ytr)
    print(f"{k_noise:>10} | {9+k_noise:>9} | {m.score(Xn_tr, ytr)*100:>10.2f}% | "
          f"{m.score(Xn_te, yte)*100:>10.2f}%")

# 可视化: 训练/测试准确率随噪声特征数的分化
ks_show = [0, 10, 50, 200, 500]
xs_log = [9+k for k in ks_show]
tr_acc, te_acc = [], []
for k_noise in [0, 10, 50, 200, 500]:
    Xn_tr = np.hstack([Xtr, rng.standard_normal((len(Xtr), k_noise))])
    Xn_te = np.hstack([Xte, rng.standard_normal((len(Xte), k_noise))])
    m = LogisticRegression(C=1e6, max_iter=3000)
    m.fit(Xn_tr, ytr)
    tr_acc.append(m.score(Xn_tr, ytr)*100)
    te_acc.append(m.score(Xn_te, yte)*100)

plt.figure(figsize=(10.5, 5.6))
plt.plot(xs_log, tr_acc, 'o-', lw=2.4, ms=8, color='#E91E63', label='训练集准确率')
plt.plot(xs_log, te_acc, 's-', lw=2.4, ms=8, color='#2196F3', label='测试集准确率')
base_te = max(yte.mean(), 1-yte.mean())*100
plt.axhline(base_te, color='gray', linestyle='--', lw=1.5, label=f'测试基准 {base_te:.1f}%')
for x_, t_ in zip(xs_log, tr_acc):
    plt.annotate(f'{t_:.1f}%', xy=(x_, t_), xytext=(0, 9),
                 textcoords='offset points', ha='center', fontsize=10, color='#E91E63')
for x_, t_ in zip(xs_log, te_acc):
    plt.annotate(f'{t_:.1f}%', xy=(x_, t_), xytext=(0, -16),
                 textcoords='offset points', ha='center', fontsize=10, color='#2196F3')
plt.xscale('log')
plt.xlabel('总特征数 (9个真实因子 + k个纯噪声)', fontsize=12)
plt.ylabel('准确率 (%)', fontsize=12)
plt.title('过拟合现场: 训练准确率飞升, 测试准确率原地踏步', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 往面板里掺入纯噪声特征: 训练/测试准确率的分道扬镳 ===
   噪声特征数 |    总特征数 |     训练准确率 |     测试准确率
         0 |         9 |      58.52% |      51.12%
        10 |        19 |      57.94% |      51.25%
        50 |        59 |      60.77% |      51.88%
       200 |       209 |      67.10% |      50.88%
       500 |       509 |      78.26% |      51.75%
```

![过拟合实验: 训练准确率随噪声特征数从58.5%飞升到78.3%, 测试准确率始终贴着51%基准线](images/ch28_fig2_overfitting.png)

**观察**:

1. **训练与测试的分道扬镳**: 塞入 500 个纯噪声后, 模型在训练集上的"准确率"冲到 78.26%, 在测试集上纹丝不动。这 27 个百分点的提升**全部来自背诵噪声**, 对未来一分钱价值都没有。
2. **这就是偏差-方差权衡的方差端**: 特征越多、模型越灵活, 对训练数据的记忆能力越强 (方差越大), 但学到真规律的能力有天花板 (偏差不变)。测试误差 = 偏差² + 方差 + 噪声——方差膨胀是过拟合的数学身份。
3. **量化研究的警示**: 每一个被试过的因子、每一组被调过的参数, 都是变相的"噪声特征"。回测框架必须像本实验一样保留一块模型从未接触的净土。

## 28.4 正则化: 给优化问题加一条缰绳

### 28.4.1 目标函数家族

正则化的思路: 既然模型自由度太大, 就在损失函数里给参数大小本身加一项"税"。

**它解决什么问题**: 第28.3节已经看到, 无正则的逻辑回归在500个噪声特征面前会疯狂过拟合。正则化的本质是给优化器加一个约束: "你可以拟合训练数据, 但系数不能太大"。L1 (Lasso) 的独特之处在于它能产生**恰好为零**的系数——自动完成特征选择。

$$
\min_w\ \underbrace{\frac{1}{n}\sum_{i=1}^n \big(y_i - f(w,x_i)\big)^2}_{\text{数据拟合项}} + \lambda\,\Omega(w)
$$

三种经典配方:

| 名称 | $\Omega(w)$ | 目标函数 | 效果 |
|------|------------|---------|------|
| Ridge (L2) | $\|w\|_2^2 = \sum_j w_j^2$ | $\|y-Xw\|^2 + \lambda\|w\|_2^2$ | 系数整体缩小, 不为零 |
| Lasso (L1) | $\|w\|_1 = \sum_j \|w_j\|$ | $\|y-Xw\|^2 + \lambda\|w\|_1$ | 部分系数**精确压到零**, 自动选特征 |
| Elastic Net | 混合 | $\lambda\big[\alpha\|w\|_1 + (1-\alpha)\|w\|_2^2\big]$ | 两者折中 |

Lasso 能产生稀疏解的几何原因: L1 的等高线是有棱角的菱形, 优化路径容易撞在角上——而角意味着某些坐标恰好为零; L2 的圆形等高线则几乎不会精确交于坐标轴。

### 28.4.2 贝叶斯解释: 先验即缰绳

正则化不是拍脑袋发明的罚款, 它有严格的概率语义。贝叶斯 MAP (最大后验) 估计为

$$
\hat{w} = \arg\max_w\ \underbrace{\ln P(y|w,X)}_{\text{似然}} + \underbrace{\ln P(w)}_{\text{先验}}
$$

- 给 $w$ 施加**高斯先验** $w_j \sim N(0, \tau^2)$: $-\ln P(w) \propto \|w\|_2^2/(2\tau^2)$——正是 Ridge;
- 给 $w$ 施加**拉普拉斯先验** (尖峰厚肩, 大多数 $w_j$ 本就该接近零): $-\ln P(w) \propto \|w\|_1/\tau$——正是 Lasso。

$\lambda$ 的身份由此揭晓: **它是"数据说话强度"与"先验信念强度"的比值** (噪声方差/先验方差)。数据少或噪声大时应当更相信先验——这正是"小样本要强正则"的数学出处。

### 28.4.3 Lasso 因子筛选实战

L1 的稀疏性让它成为天然的因子筛选器: 正则越强, 幸存的因子越少, 而"最后被淘汰的"就是数据最坚持的因子。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def build_panel():
    df = pd.read_csv('data/stock_data_50_20210601_20260531.csv')
    df['time'] = pd.to_datetime(df['time'])
    px = df.pivot(index='time', columns='thscode', values='close').sort_index().ffill()
    high = df.pivot(index='time', columns='thscode', values='high').sort_index().ffill()
    low = df.pivot(index='time', columns='thscode', values='low').sort_index().ffill()
    volu = df.pivot(index='time', columns='thscode', values='volume').sort_index().ffill()
    mcap = df.pivot(index='time', columns='thscode', values='market_cap').sort_index().ffill()

    s_idx = pd.Series(px.index)
    mon_end = s_idx.groupby(pd.to_datetime(s_idx).dt.to_period('M')).max().values
    pm = px.loc[mon_end]

    feats = {
        '动量12-1':  px.shift(21)/px.shift(252) - 1,
        '反转3月':   -(px/px.shift(63) - 1),
        '反转1月':   -(px/px.shift(21) - 1),
        '低波动':    -(px.pct_change().rolling(60).std()*np.sqrt(252)),
        '振幅':      (high/low - 1).rolling(20).mean(),
        '流动性':    np.log(volu.rolling(20).mean()),
        '市值':      np.log(mcap.loc[mon_end]).reindex(pm.index),
        '距52周高点': px/px.rolling(252).max() - 1,
        '价格位置':   ((px - px.rolling(252).min()) /
                    (px.rolling(252).max() - px.rolling(252).min())),
    }
    frames = []
    for name, daily in feats.items():
        z = daily.loc[mon_end].apply(lambda s: (s-s.mean())/s.std(), axis=1)
        frames.append(z.stack().rename(name))
    X = pd.concat(frames, axis=1)
    y_ret = pm.pct_change().shift(-1).stack().rename('ret')
    data = X.join(y_ret).dropna()
    data['up'] = (data['ret'] > 0).astype(int)
    return data.reset_index().rename(columns={'time': 'date', 'thscode': 'code'})

FEATS = ['动量12-1','反转3月','反转1月','低波动','振幅','流动性','市值','距52周高点','价格位置']

panel = build_panel()
split_date = pd.Timestamp('2025-01-01')
tr = panel[pd.to_datetime(panel['date']) < split_date]
scaler = StandardScaler().fit(tr[FEATS].values)
Xtr, ytr = scaler.transform(tr[FEATS].values), tr['up'].values

print("=== L1 正则化的系数路径: 谁先入选, 谁被淘汰 ===")
C_grid = np.logspace(-2, 1.4, 30)
paths = {f: [] for f in FEATS}
nnz = []
for C in C_grid:
    m = LogisticRegression(C=float(C), solver='liblinear', l1_ratio=1, max_iter=5000)
    m.fit(Xtr, ytr)
    for i, f in enumerate(FEATS):
        paths[f].append(m.coef_[0][i])
    nnz.append(int((np.abs(m.coef_[0]) > 1e-8).sum()))

show_idx = [0, 4, 8, 12, 16, 20, 29]
print(f"{'C(=1/lam)':>9} | " + ' | '.join(f'{f}' for f in FEATS[:5]))
print('-'*70)
for j in show_idx:
    row = ' | '.join(f'{paths[f][j]:+.2f}' for f in FEATS[:5])
    print(f"{C_grid[j]:>9.3f} | {row} | 非零={nnz[j]}")

entry_C = {}
for f in FEATS:
    nz = [C_grid[j] for j in range(len(C_grid)) if abs(paths[f][j]) > 1e-8]
    entry_C[f] = min(nz) if nz else np.inf
order = sorted(entry_C, key=lambda f: entry_C[f])
print()
print("Lasso 入选顺序 (随正则减弱, 系数最先变非零的因子排前面):")
for rank, f in enumerate(order, 1):
    print(f"  {rank}. {f:<8} (入选于 C={entry_C[f]:.3f})")

# 可视化: 系数路径与复杂度曲线
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
cmap9 = dict(zip(FEATS, ['#1f77b4','#d62728','#ff7f0e','#2ca02c','#9467bd',
                          '#8c564b','#e377c2','#7f7f7f','#bcbd22']))
ax = axes[0]
for f in FEATS:
    lw_line = 3.0 if f in ('低波动', '反转3月') else 1.4
    ax.plot(C_grid, paths[f], lw=lw_line, label=f, color=cmap9[f])
ax.set_xscale('log')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('C = 1/λ (左端=强正则, 右端=弱正则)', fontsize=12)
ax.set_ylabel('逻辑回归系数', fontsize=12)
ax.set_title('L1 系数路径: 强正则下只留下少数幸存者', fontsize=12)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(C_grid, nnz, 'o-', lw=2, ms=6, color='#2196F3')
ax.set_xscale('log')
ax.set_xlabel('C = 1/λ', fontsize=12)
ax.set_ylabel('非零系数个数 (共9个)', fontsize=12)
ax.set_title('正则强度决定模型复杂度: Lasso 自动做特征选择', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== L1 正则化的系数路径: 谁先入选, 谁被淘汰 ===
C(=1/lam) | 动量12-1 | 反转3月 | 反转1月 | 低波动 | 振幅
----------------------------------------------------------------------
    0.010 | +0.00 | +0.00 | +0.00 | +0.00 | +0.00 | 非零=0
    0.029 | +0.00 | +0.00 | +0.00 | +0.07 | +0.00 | 非零=1
    0.087 | +0.00 | +0.02 | +0.00 | +0.12 | +0.00 | 非零=3
    0.255 | -0.04 | +0.08 | -0.02 | +0.15 | +0.03 | 非零=9
    0.751 | -0.06 | +0.10 | -0.01 | +0.20 | +0.10 | 非零=9
    2.212 | -0.07 | +0.10 | -0.01 | +0.22 | +0.12 | 非零=9
   25.119 | -0.08 | +0.11 | -0.01 | +0.23 | +0.13 | 非零=9

Lasso 入选顺序 (随正则减弱, 系数最先变非零的因子排前面):
  1. 低波动      (入选于 C=0.017)
  2. 反转3月     (入选于 C=0.066)
  3. 流动性      (入选于 C=0.066)
  4. 反转1月     (入选于 C=0.114)
  5. 距52周高点   (入选于 C=0.114)
  6. 动量12-1   (入选于 C=0.149)
  7. 振幅       (入选于 C=0.195)
  8. 价格位置     (入选于 C=0.195)
  9. 市值       (入选于 C=0.255)
```

![L1系数路径: 左图-各因子系数随C的变化, 低波动(绿粗线)最先入场且幅度最大; 右图-非零系数个数从0爬升到9](images/ch28_fig1_lasso_paths.png)

**观察**:

1. **入选顺序是一份"数据信任榜"**: 最强正则时全场归零, 放松缰绳后**低波动第一个进场**, 反转3月紧随其后——与第27章 IC 筛选、28.2 节逻辑回归系数的排序高度一致。三条独立证据链指向同一批因子。
2. **Lasso 的筛选是"自动"且"连续"的**: 不需要人工设定"保留几个因子", 正则强度 $\lambda$ 本身就是旋钮。图右的非零计数曲线展示了模型复杂度如何被一个参数平滑控制。
3. **对照 28.3 节**: 同样面对噪声特征的威胁, 无正则模型全盘接收, 而 L1 会把噪声系数压成精确的零。**正则化就是把先验知识 ("大部分因子没用") 写进优化目标**。

## 28.5 交叉验证与时间序列泄漏

调参 (比如选 $\lambda$) 不能用测试集——否则测试集就成了训练的一部分。标准答案是**交叉验证**: 把训练数据再切成 K 折, 轮流留一折做验证:

$$
CV(k) = \frac{1}{k}\sum_{j=1}^{k}\ \text{loss}\big(\hat{f}^{(-j)}, D_j\big)
$$

但金融数据有一条铁律: **随机打乱的 K 折会把未来的信息漏进过去**。2026 年的收益模式会参与训练、然后去"验证"2023 年的数据——验证分数虚高, 选出的参数看似很好实则无效。正确姿势是**走前验证** (walk-forward): 只用 $t$ 之前的数据定参, 在 $t$ 上检验, 逐月滚动推进。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

def build_panel():
    df = pd.read_csv('data/stock_data_50_20210601_20260531.csv')
    df['time'] = pd.to_datetime(df['time'])
    px = df.pivot(index='time', columns='thscode', values='close').sort_index().ffill()
    high = df.pivot(index='time', columns='thscode', values='high').sort_index().ffill()
    low = df.pivot(index='time', columns='thscode', values='low').sort_index().ffill()
    volu = df.pivot(index='time', columns='thscode', values='volume').sort_index().ffill()
    mcap = df.pivot(index='time', columns='thscode', values='market_cap').sort_index().ffill()

    s_idx = pd.Series(px.index)
    mon_end = s_idx.groupby(pd.to_datetime(s_idx).dt.to_period('M')).max().values
    pm = px.loc[mon_end]

    feats = {
        '动量12-1':  px.shift(21)/px.shift(252) - 1,
        '反转3月':   -(px/px.shift(63) - 1),
        '反转1月':   -(px/px.shift(21) - 1),
        '低波动':    -(px.pct_change().rolling(60).std()*np.sqrt(252)),
        '振幅':      (high/low - 1).rolling(20).mean(),
        '流动性':    np.log(volu.rolling(20).mean()),
        '市值':      np.log(mcap.loc[mon_end]).reindex(pm.index),
        '距52周高点': px/px.rolling(252).max() - 1,
        '价格位置':   ((px - px.rolling(252).min()) /
                    (px.rolling(252).max() - px.rolling(252).min())),
    }
    frames = []
    for name, daily in feats.items():
        z = daily.loc[mon_end].apply(lambda s: (s-s.mean())/s.std(), axis=1)
        frames.append(z.stack().rename(name))
    X = pd.concat(frames, axis=1)
    y_ret = pm.pct_change().shift(-1).stack().rename('ret')
    data = X.join(y_ret).dropna()
    data['up'] = (data['ret'] > 0).astype(int)
    return data.reset_index().rename(columns={'time': 'date', 'thscode': 'code'})

FEATS = ['动量12-1','反转3月','反转1月','低波动','振幅','流动性','市值','距52周高点','价格位置']

panel = build_panel()
split_date = pd.Timestamp('2025-01-01')
tr = panel[pd.to_datetime(panel['date']) < split_date]
te = panel[pd.to_datetime(panel['date']) >= split_date]
scaler = StandardScaler().fit(tr[FEATS].values)
Xall, yall = scaler.transform(panel[FEATS].values), panel['up'].values

C_list = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]

# 方法A: 天真的随机K折 -- 在整个面板(含未来!)上打乱交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_means = {}
for C in C_list:
    accs = []
    for tr_idx, va_idx in skf.split(Xall, yall):
        m = LogisticRegression(C=C, solver='liblinear', l1_ratio=1, max_iter=3000)
        m.fit(Xall[tr_idx], yall[tr_idx])
        accs.append(m.score(Xall[va_idx], yall[va_idx]))
    cv_means[C] = np.mean(accs)
best_C_naive = max(cv_means, key=cv_means.get)

# 方法B: 走前验证 -- 内层尾段选C, 外层只测真正的未来月份
months_all = sorted(panel['date'].unique())
test_months = [d for d in months_all if pd.Timestamp(d) >= split_date]
oos_default, oos_wf = [], []
for tm in test_months:
    sub_tr = panel[pd.to_datetime(panel['date']) < tm]
    sub_te = panel[pd.to_datetime(panel['date']) == tm]
    sc = StandardScaler().fit(sub_tr[FEATS].values)
    Xt, yt = sc.transform(sub_tr[FEATS].values), sub_tr['up'].values
    Xv, yv = sc.transform(sub_te[FEATS].values), sub_te['up'].values

    m_def = LogisticRegression(C=1e6, max_iter=3000)   # 近似无正则
    m_def.fit(Xt, yt)
    oos_default.append(m_def.score(Xv, yv))

    n_inner = int(len(yt)*0.85)
    best_acc, best_C = -1, None
    for C in C_list:
        m_in = LogisticRegression(C=C, solver='liblinear', l1_ratio=1, max_iter=3000)
        m_in.fit(Xt[:n_inner], yt[:n_inner])
        acc = m_in.score(Xt[n_inner:], yt[n_inner:])
        if acc > best_acc:
            best_acc, best_C = acc, C
    m_wf = LogisticRegression(C=best_C, solver='liblinear', l1_ratio=1, max_iter=3000)
    m_wf.fit(Xt, yt)
    oos_wf.append(m_wf.score(Xv, yv))

n_test = len(test_months)
print("=== 三种调参方式的样本外表现 (逐月滚动, 共%d个月) ===" % n_test)
print(f"{'方式':<32} | {'OOS准确率':>9}")
print('-'*48)
print(f"{'默认 C=1 (不调参)':<31} | {np.mean(oos_default)*100:>8.2f}%")
print(f"{'走前窗口调参 (无泄漏)':<31} | {np.mean(oos_wf)*100:>8.2f}%")
print()
print(f"天真随机K折在'全样本'上的最优 C = {best_C_naive}, 其CV准确率 = {cv_means[best_C_naive]*100:.2f}%")
print("(该数字混合了未来月份的信息, 因此系统性偏高 -- 这就是泄漏)")

**运行结果**:

```
=== 三种调参方式的样本外表现 (逐月滚动, 共16个月) ===
方式                               |    OOS准确率
------------------------------------------------
默认 C=1 (不调参)                    |    51.12%
走前窗口调参 (无泄漏)                    |    50.88%

天真随机K折在'全样本'上的最优 C = 0.1, 其CV准确率 = 55.87%
(该数字混合了未来月份的信息, 因此系统性偏高 -- 这就是泄漏)
```

**观察**:

1. **55.87% vs 50.88%——四个半点的幻觉**: 天真 K 报出的 CV 准确率比诚实的走前验证高出近 5 个百分点。这 5 个点不来自更好的模型, 而来自**时间机器**: 未来数据参与了调参。
2. **诚实的结果反而平淡**: 走前调参 (50.88%) 与干脆不调参 (51.12%) 几乎无差别。信号本身就弱时, 调参的空间自然有限——承认这一点比假装调出了增益更有价值。
3. **实务清单**: 金融 ML 的标准配置是 purged K-fold / embargo (剔除标签重叠期并在折间留隔离带); 无论用哪种, 判据只有一条——**任何用于评估的数据不得以任何形式参与训练或调参**。

## 28.6 Ridge 与协方差收缩: 第14章的回响

Ridge 的思想不止用于回归。第14章我们见过: 用短窗口估出的股票协方差矩阵病态 (条件数巨大), 直接拿去做组合优化会放大噪声。Ridge 给出最直接的修复——**给对角线加一条缰绳**:

$$
\Sigma_\lambda = \Sigma + \lambda\bar{\sigma}^2 I
$$

每加一分 $\lambda$, 条件数就降一分。它与第14章的 Ledoit-Wolf 收缩殊途同归: LW 把 $\Sigma$ 往缩放单位阵方向压缩, Ridge 是它的对角特例。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf

df5 = pd.read_csv('data/stock_data_50_20210601_20260531.csv')
df5['time'] = pd.to_datetime(df5['time'])
px5 = df5.pivot(index='time', columns='thscode', values='close').sort_index().ffill()
rets_d = px5.pct_change().iloc[1:]        # 首行无前值, 整行 NaN

train_w, test_w = 480, 240
R_tr = rets_d.iloc[:train_w].dropna(axis=1, how='any')
codes_ok = R_tr.columns
R_te = rets_d.iloc[train_w:train_w+test_w][codes_ok].dropna()

S = R_tr.cov().values
n_assets = S.shape[0]
cond_S = np.linalg.cond(S)

lw = LedoitWolf().fit(R_tr.values)
S_lw = lw.covariance_
cond_lw = np.linalg.cond(S_lw)

ones = np.ones(n_assets)
def oos_vol(Sm):
    """给定协方差估计, 构建最小方差组合并计算样本外实现波动"""
    w = np.linalg.solve(Sm, ones)
    w = w/w.sum()
    return float(np.sqrt(w @ np.cov(R_te.values.T) @ w)*np.sqrt(252))

print("=== Ridge 思想的矩阵版: 给协方差加一条对角缰绳 ===")
print(f"资产数={n_assets}, 训练窗口={train_w}天, 测试窗口={test_w}天")
print()
print(f"{'矩阵':>12} | {'条件数':>8} | {'最小方差组合OOS年化波动':>18}")
print('-'*52)
print(f"{'样本协方差':>11} | {cond_S:>10.0f} | {oos_vol(S)*100:>19.2f}%")
diag_scale = np.trace(S)/n_assets
for lam in [0.05, 0.1, 0.3, 1.0]:
    Sm = S + lam*diag_scale*np.eye(n_assets)
    print(f"{'Ridge λ='+str(lam):>11} | {np.linalg.cond(Sm):>10.0f} | {oos_vol(Sm)*100:>19.2f}%")
print(f"{'Ledoit-Wolf':>12} | {cond_lw:>10.0f} | {oos_vol(S_lw)*100:>19.2f}%")

# 可视化: 条件数与OOS波动随收缩强度的变化
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

lams = np.logspace(-3, 0.5, 40)
conds, vols = [], []
for lam in lams:
    Sm = S + lam*diag_scale*np.eye(n_assets)
    conds.append(np.linalg.cond(Sm))
    vols.append(oos_vol(Sm))
lam_star = lams[int(np.argmin(vols))]
v_samp, v_lw = oos_vol(S), oos_vol(S_lw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].loglog(lams, conds, lw=2.4, color='#2196F3',
               label=r'条件数 $\kappa(\Sigma+\lambda I)$')
axes[0].axhline(cond_S, color='#999999', ls='--', lw=1.5,
                label=f'样本协方差 {cond_S:.0f}')
axes[0].axhline(cond_lw, color='#4CAF50', ls=':', lw=2,
                label=f'Ledoit-Wolf {cond_lw:.0f}')
axes[0].set_xlabel(r'收缩强度 $\lambda$', fontsize=12)
axes[0].set_ylabel('条件数', fontsize=12)
axes[0].set_title('对角缰绳让病态矩阵变得温顺', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, which='both')

axes[1].semilogx(lams, vols, lw=2.4, color='#E91E63',
                 label='Ridge 收缩后的 OOS 波动')
axes[1].axhline(v_samp, color='#999999', ls='--', lw=1.5,
                label=f'无收缩 {v_samp:.2f}%')
axes[1].axhline(v_lw, color='#4CAF50', ls=':', lw=2,
                label=f'Ledoit-Wolf {v_lw:.2f}%')
axes[1].scatter([lam_star], [min(vols)], s=110, color='#333333', zorder=5)
axes[1].annotate(f'最优 λ*={lam_star:.3f}\n{min(vols):.2f}%',
                 xy=(lam_star, min(vols)), xytext=(lam_star*2.2, min(vols)-0.55),
                 fontsize=10, arrowprops=dict(arrowstyle='->', color='#333333'))
axes[1].set_xlabel(r'收缩强度 $\lambda$', fontsize=12)
axes[1].set_ylabel('最小方差组合 OOS 年化波动', fontsize=12)
axes[1].set_title('适度收缩降低样本外风险; 缩过头连信号一起缩没', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

**运行结果**:

```
=== Ridge 思想的矩阵版: 给协方差加一条对角缰绳 ===
资产数=50, 训练窗口=480天, 测试窗口=240天

          矩阵 |      条件数 |      最小方差组合OOS年化波动
----------------------------------------------------
      样本协方差 |        274 |               11.44%
Ridge λ=0.05 |        138 |               10.97%
Ridge λ=0.1 |         93 |               11.03%
Ridge λ=0.3 |         40 |               11.78%
Ridge λ=1.0 |         14 |               13.51%
 Ledoit-Wolf |        160 |               11.01%
```

![Ridge收缩: 左图-条件数随lambda单调下降(274降到两位数); 右图-OOS波动呈U型, 最优点λ*=0.04附近, 过度收缩反而变差](images/ch28_fig3_ridge_shrinkage.png)

**观察**:

1. **条件数一路下降, 风险却呈 U 型**: $\lambda$ 从 0 加大, 样本外波动先降后升 (11.44% → 10.97% → 13.51%)。适度收缩滤掉的是采样噪声; 过度收缩连真实的相关结构也一起抹平——**正则化的剂量本身就是需要学习的超参数**。
2. **Ledoit-Wolf 是自动挡**: 不用手调 $\lambda$, 数据驱动地给出收缩强度, 表现 (11.01%) 落在本例最优 Ridge 附近。第14章的条件数实验 (519→233) 与这里的 274→138 是同一枚硬币的两面。
3. **统一视角**: 回归里的 Ridge、协方差里的收缩、贝叶斯里的先验, 是同一思想的三种化身——**当数据不足以支撑完全自由的模型时, 用可控的偏见换取稳定的预测**。

## 28.7 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

1. **Sigmoid**: $p = \sigma(z) = 1/(1+e^{-z})$, $z = w^\top x + b$
2. **交叉熵损失**: $L = -\frac{1}{n}\sum_i \big[y_i\ln p_i + (1-y_i)\ln(1-p_i)\big]$; 梯度 $\partial L_i/\partial z_i = p_i - y_i$
3. **Ridge**: $\min_w \|y-Xw\|^2 + \lambda\|w\|_2^2$; 闭式解 $\hat{w} = (X^\top X + \lambda I)^{-1}X^\top y$
4. **Lasso**: $\min_w \|y-Xw\|^2 + \lambda\|w\|_1$; 稀疏解, 用于特征筛选
5. **Elastic Net**: $\lambda\big[\alpha\|w\|_1 + (1-\alpha)\|w\|_2^2\big]$
6. **贝叶斯解释 (MAP)**: 高斯先验 $\leftrightarrow$ L2; 拉普拉斯先验 $\leftrightarrow$ L1; $\lambda \approx$ 噪声方差/先验方差
7. **K 折交叉验证**: $CV = \frac{1}{k}\sum_j \text{loss}(\hat{f}^{(-j)}, D_j)$; 时间序列须用走前/purged 版本防泄漏
8. **协方差 Ridge**: $\Sigma_\lambda = \Sigma + \lambda\bar{\sigma}^2 I$; 条件数随 $\lambda$ 单调下降
9. **偏差-方差分解**: 测试误差 $\approx$ 偏差$^2$ + 方差 + 不可约噪声

## 28.8 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 逻辑回归 | sigmoid + 交叉熵, MLE 即训练 | 因子→涨跌概率的标准载体 |
| 过拟合实验 | 噪声特征让训练/测试分道扬镳 | 训练进步 ≠ 真本事 |
| Ridge/Lasso/ElasticNet | L2 缩放 / L1 选择 / 混合 | 高维小样本的生存工具 |
| 贝叶斯解释 | 先验即正则, $\lambda$=数据vs信念 | 正则强度有了物理意义 |
| Lasso 入选顺序 | 数据最坚持的因子活到最后 | 与 IC 筛选互为独立证据链 |
| 走前验证 | 只用过去调参, 只对未来评分 | 防时间泄漏的生命线 |
| 协方差收缩 | $\Sigma + \lambda I$ 驯服条件数 | 组合优化的稳定器 (ch14 回响) |

**最后一句话**: 本章的全部技术都服务于一个朴素的信念——**宁可模型笨一点, 也不要它背错书**。正则化、交叉验证、走前检验, 本质上都是在给"我想赢"的冲动加装刹车。刹车装好了, 剩下的交给数据。

## 28.9 练习题

### 数学推导

**题1——交叉熵的梯度**:

(a) 证明 sigmoid 的导数满足 $\sigma'(z) = \sigma(z)(1-\sigma(z))$。

(b) 对单个样本的交叉熵 $L = -[y\ln p + (1-y)\ln(1-p)]$, 利用 (a) 证明 $\partial L/\partial z = p - y$。

(c) 解释该梯度形式为什么使交叉熵 + sigmoid 成为"天生一对", 而 MSE 配 sigmoid 会出现什么问题 (提示: 考虑 $p \to 0, y = 1$ 时两种损失的梯度行为)。

**题2——正则化的贝叶斯推导**:

(a) 设先验 $w_j \sim N(0, \tau^2)$ 相互独立, 证明 $-\ln P(w) = \frac{1}{2\tau^2}\|w\|_2^2 + \text{const}$, 因此高斯先验的 MAP 等价于 Ridge (其中 $\lambda = \sigma_\epsilon^2/\tau^2$, $\sigma_\epsilon^2$ 为噪声方差)。

(b) 拉普拉斯先验 $P(w_j) \propto e^{-|w_j|/b}$ 对应哪个正则项? 写出对应关系。

(c) 由 (a) 中 $\lambda$ 的表达式说明: 为什么"数据噪声更大"和"更相信系数接近零"都会导致更强的有效正则?

**题3——Ridge 的 SVD 视角**:

(a) 设 $X = U\Sigma V^\top$ (奇异值分解), 证明 Ridge 解可写成 $\hat{w} = \sum_j \frac{\sigma_j}{\sigma_j^2 + \lambda} u_j^\top y\ v_j$, 并说明当 $\lambda = 0$ 时退化为普通 OLS。

(b) 由 (a) 说明: OLS 对小奇异值方向 ($\sigma_j$ 小) 极其敏感, 而 Ridge 把这些方向的放大系数从 $1/\sigma_j$ 压到 $\sigma_j/(\sigma_j^2+\lambda)$。

(c) 对照本章协方差实验: 把 $\Sigma + \lambda I$ 的特征值写为 $\mu_j + \lambda$, 证明条件数不超过 $(\mu_{\max}+\lambda)/(\mu_{\min}+\lambda)$ 且随 $\lambda$ 单调递减。

### 编程实践

**题1——Elastic Net 网格搜索**: 在本章面板上, 对 `l1_ratio ∈ {0.1, 0.3, 0.5, 0.7, 0.9}` 与 `C ∈ {0.03, 0.1, 0.3}` 共 15 个组合做走前验证 (参考 28.5 节代码), 输出 OOS 准确率矩阵; 找出最优组合并检查其非零系数集合在不同验证月份中的稳定性。

**题2——Lasso 筛选后的分层回测**: 取 28.4.3 节 Lasso 入选顺序的前 5 个因子 (低波动、反转3月、流动性、反转1月、距52周高点), 将它们的 z-score 等权平均合成新因子, 重跑第27章的 Rank IC 与五分层回测。对比合成因子与单一反转3月的 IR 和多空夏普, 并讨论: 多因子合成的分散化收益 vs 数据窥探风险, 你更担心哪一个?

## 28.10 参考文献

1. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.（第3-4章线性方法与分类, 第7章模型选择与交叉验证——本章的理论骨架）

2. Tibshirani, R. (1996). "Regression Shrinkage and Selection via the Lasso." *Journal of the Royal Statistical Society: Series B*, 58(1), 267-288.（Lasso 的开山之作）

3. Hoerl, A. E., & Kennard, R. W. (1970). "Ridge Regression: Biased Estimation for Nonorthogonal Problems." *Technometrics*, 12(1), 55-67.（Ridge 回归的原始文献）

4. López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley.（金融机器学习中泄漏问题的系统论述, purged K-fold 的出处）